In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
UST1=pd.read_csv("")

In [ ]:
UST1.head()

In [ ]:
pd.options.display.max_columns = None

In [ ]:
UST1.columns

In [ ]:
UST1.shape

In [ ]:
#import cleaned up dataset
UST2=pd.read_csv('/Users/jojoli/Documents/棒呆留学申请资料/2nd-3rd year/Columbia University夏校/Columbia Classes/Big Data/Datasets/UST2.csv')

In [ ]:
UST2.head(10)

In [ ]:
UST2.shape

In [ ]:
#Machine learning model with actual data
UST=pd.read_csv('/Users/jojoli/Documents/棒呆留学申请资料/2nd-3rd year/Columbia University夏校/Columbia Classes/Big Data/Datasets/UST22.csv')

In [ ]:
UST.head(10)

In [ ]:
UST.shape

In [ ]:
UST.columns

In [ ]:
UST.info()

In [ ]:
sns.heatmap(UST.corr(), fmt = ".2f")

In [ ]:
sns.distplot(UST['Amount'], kde=True)

In [ ]:
sns.distplot(UST["Amount"], kde=True)

In [ ]:
ust = UST[['Amount', 'Expected Revenue', 'Probability', 'Stage Number', 'New Business', 'MSA Signed', 'Account Type']]

In [ ]:
sns.pairplot(ust, hue='Account Type')

In [ ]:
UST.Amount.mean()

In [ ]:
UST.Amount.max()

In [ ]:
UST[UST.Amount == 91000000.0]

In [ ]:
UST.Amount.min()

In [ ]:
UST[UST.Amount == -900960.0]

In [ ]:
UST['Expected Revenue'].min()

In [ ]:
UST['Expected Revenue'].max()

In [ ]:
UST[UST['Expected Revenue'] == 39600000.0]

In [ ]:
ust2 = UST[['Amount', 'Expected Revenue', 'Probability', 'Stage Number', 'New Business', 'MSA Signed', 'Deliverable Type']]

In [ ]:
sns.pairplot(ust2, hue='Deliverable Type')

In [ ]:
#following teacher's steps

In [ ]:
UST.describe()

In [ ]:
UST['Amount'].mean(),UST['Amount'].median()

In [ ]:
x=UST['Stage Number']
y=UST['Expected Revenue']
plt.scatter(x,y)

In [ ]:
#plot to see the distribtion of revenues vs stage number
rng=np.random.RandomState(0)
x=UST2['Stage Number']
y=UST2['Expected Revenue']
colors=rng.rand(100)
sizes=1000*rng.rand(100)
plt.scatter(x,y,c=colors,s=sizes,alpha=.3)
plt.colorbar()

In [ ]:
# Find all correlations and sort 
correlations_data = UST.corr()['Probability'].sort_values()

# Print the correlations
print(correlations_data.head(15), '\n')

In [ ]:
#new business negatively correlated: if new, then less probability of getting the project
#higher the amount is, the lower the probability of getting the project
#probability increases as stage number increases
#they are correct from business perspective

In [ ]:
# Find all correlations and sort
correlations_data2 = UST2.corr()['Probability'].sort_values()

# Print the most negative correlations
print(correlations_data2.head(15), '\n')

In [ ]:
#Looking at the missing data

def missingdata(data):
    total = data.isnull().sum().sort_values(ascending = False)
    percent = (data.isnull().sum()/data.isnull().count()*100).sort_values(ascending = False)
    ms=pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
    ms= ms[ms["Percent"] > 0]
    f,ax =plt.subplots(figsize=(8,6))
    plt.xticks(rotation='90')
    fig=sns.barplot(ms.index, ms["Percent"],color="green",alpha=0.8)
    plt.xlabel('Features', fontsize=15)
    plt.ylabel('Percent of missing values', fontsize=15)
    plt.title('Percent missing data by feature', fontsize=15)
    return ms
missingdata(UST)

In [ ]:
#Remove "Service Category"

In [ ]:
drop_column = ['Service Category']
UST.drop(drop_column, axis=1, inplace = True)

In [ ]:
#fill the null values
UST['Billing Type'].fillna(UST['Billing Type'].mode()[0], inplace = True)
UST['Deliverable Type'].fillna(UST['Deliverable Type'].mode()[0], inplace = True)
UST['Account Type'].fillna(UST['Account Type'].mode()[0], inplace = True)
UST['Unsolicited Proposal'].fillna(UST['Unsolicited Proposal'].mode()[0], inplace = True)

In [ ]:
# check if the missing data is cleaned
print('check the nan value in the UST data')
print(UST.isnull().sum())

In [ ]:
#Feature Engineering

In [ ]:
#convert categorical variables to numerical ones: dummy them

In [ ]:
sample=UST['Billing Type']
print(sample.unique())

In [ ]:
sample

In [ ]:
sampledf = pd.get_dummies(sample, columns = ["Billing Type"], prefix=["Bill"])

In [ ]:
sampledf.head(10)

In [ ]:
#convert all categorical variables to numerical values
all_data = UST
import re

traindf=UST

traindf = pd.get_dummies(traindf, columns = ["Opportunity Owner","Account Name","Stage","Service Line","Stage Number","Account Type","Billing Type","Unsolicited Proposal","Deliverable Type"],
                             prefix=["Owner","Account","Stg","SvcLn","StgNum","Acctype","Billngtype","UnSolProp","DelType"])

In [ ]:
traindf.head()

In [ ]:
traindf.shape
#about 3.5 million values now

In [ ]:
traindf.dtypes

In [ ]:
#look at the new correlations
correlations_traindf = traindf.corr()['Probability'].sort_values()
print(correlations_traindf.head(15), '\n')

In [ ]:
#the first two make sense

In [ ]:
print(correlations_traindf.tail(15), '\n')

## Modeling

In [ ]:
from sklearn.model_selection import train_test_split #for split the data
from sklearn.metrics import accuracy_score  #for accuracy_score
from sklearn.model_selection import KFold #for K-fold cross validation
from sklearn.model_selection import cross_val_score #score evaluation
from sklearn.model_selection import cross_val_predict #prediction
from sklearn.model_selection import GridSearchCV # for Hyper parameter tuning
from sklearn.metrics import confusion_matrix #for confusion matrix
from sklearn import metrics

In [ ]:
all_features = traindf.drop("Probability",axis=1)
Targeted_feature = traindf["Probability"]

# total of 6891 records in dataset
# Divide the data set into two- 70% for train and 30% for test

#X_train,X_test,y_train,y_test = train_test_split(all_features,Targeted_feature,test_size=0.3)  
X_train,X_test,y_train,y_test = train_test_split(all_features,Targeted_feature,test_size=0.3,random_state=42) 

# Check the train test data set shape
X_train.shape,X_test.shape,y_train.shape,y_test.shape

In [ ]:
# Train the logistic regression model with training data

from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_train, y_train)

#now it's already trained

In [ ]:
#predict the model output with test data
y_pred = logreg.predict(X_test)

print('--------------The Accuracy of the model----------------------------')
print('Accuracy of logistic regression classifier on test set: {:.8f}'.format(logreg.score(X_test, y_test)))

sns.heatmap(confusion_matrix(y_test,y_pred),annot=True,fmt='3.0f',cmap="summer")
plt.title('Confusion_matrix', y=1.05, size=15)

In [ ]:
sns.heatmap(confusion_matrix(y_test,y_pred),annot=True,fmt='3.0f',cmap="summer")
plt.title('Confusion_matrix', y=1.05, size=15)
plt.xlabel('true label')
plt.ylabel('predicted label')
#it shows false negatives and false positives

In [ ]:
print(metrics.classification_report(y_pred,y_test))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(criterion='gini', n_estimators=700,
                             min_samples_split=10,min_samples_leaf=1,
                             max_features='auto',oob_score=True,
                             random_state=1,n_jobs=-1)
model.fit(X_train,y_train)
y_pred_rm=model.predict(X_test)

In [ ]:
print(metrics.classification_report(y_pred_rm,y_test))

In [ ]:
from sklearn.metrics import confusion_matrix
mat=confusion_matrix(y_test,y_pred_rm)
sns.heatmap(mat.T,square=True,annot=True,fmt='d',cbar=False)
plt.xlabel('true label')
plt.ylabel('predicted label')

In [ ]:
print(y_pred_rm)

In [ ]:
y_pred_rm.tofile('/Users/jojoli/Documents/棒呆留学申请资料/2nd year/Columbia University夏校/Columbia Classes/Big Data/Datasets/UST22 Prediction 1.csv',sep=',')

In [ ]:
pd.DataFrame(y_test).to_csv("/Users/jojoli/Documents/棒呆留学申请资料/2nd year/Columbia University夏校/Columbia Classes/Big Data/Datasets/UST22-y_test.csv.csv")

In [ ]:
from sklearn.neural_network import MLPClassifier
model1 = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1)
model1.fit(X_train,y_train)
y_pred_rm=model1.predict(X_test)

In [ ]:
print(metrics.classification_report(y_pred_rm,y_test))

In [ ]:
#predict the model output with test data
y_pred = model1.predict(X_test)

print('--------------The Accuracy of the model----------------------------')
print('Accuracy of logistic regression classifier on test set: {:.8f}'.format(model1.score(X_test, y_test)))

sns.heatmap(confusion_matrix(y_test,y_pred),annot=True,fmt='3.0f',cmap="summer")
plt.title('Confusion_matrix', y=1.05, size=15)